In [1]:
import os
import warnings
import logging
import joblib
import pandas as pd
from prophet import Prophet

# Suppress warnings and cmdstanpy info logs
warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

print("--- LOADING DATASET ---")
data_path = 'data/7_cities_ML_ready.csv'

if not os.path.exists(data_path):
    raise FileNotFoundError(f"File not found at '{data_path}'!")

df = pd.read_csv(data_path)
df['Datetime'] = pd.to_datetime(df['Datetime'])

# Resample hourly data to daily average per city
df_daily = df.groupby(['City', pd.Grouper(key='Datetime', freq='D')])['AQI'].mean().reset_index()

cities = ['Delhi', 'Mumbai', 'Bengaluru', 'Chennai', 'Hyderabad', 'Ahmedabad', 'Kolkata']
os.makedirs('models', exist_ok=True)

print(f"\n--- TRAINING PROPHET MODELS FOR {len(cities)} CITIES ---")

for city in cities:
    # Filter city data and take recent 2 years
    city_df = df_daily[df_daily['City'] == city][['Datetime', 'AQI']].dropna()
    city_df = city_df.sort_values('Datetime').tail(730)
    city_df.columns = ['ds', 'y']
    
    # Set cap and floor for logistic growth
    city_df['cap'] = 500
    city_df['floor'] = 10
    
    # Fit Bounded Prophet Model
    model = Prophet(growth='logistic', daily_seasonality=False, yearly_seasonality=True)
    model.fit(city_df)
    
    # Save model file with city name
    file_name = f"models/prophet_{city.lower()}.pkl"
    joblib.dump(model, file_name)
    print(f"✓ Saved: {file_name}")

print("\n================ ALL MODELS TRAINED SUCCESSFULLY ================")

c:\Users\vaish\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- LOADING DATASET ---


11:25:37 - cmdstanpy - INFO - Chain [1] start processing



--- TRAINING PROPHET MODELS FOR 7 CITIES ---


11:25:38 - cmdstanpy - INFO - Chain [1] done processing
11:25:38 - cmdstanpy - INFO - Chain [1] start processing


...

✓ Saved: models/prophet_hyderabad.pkl


11:25:39 - cmdstanpy - INFO - Chain [1] done processing
11:25:39 - cmdstanpy - INFO - Chain [1] start processing


✓ Saved: models/prophet_ahmedabad.pkl


11:25:39 - cmdstanpy - INFO - Chain [1] done processing


✓ Saved: models/prophet_kolkata.pkl

================ ALL MODELS TRAINED SUCCESSFULLY ================
